# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ashoktanakanti/flyrank_ml/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

*Look before deciding: distributions of my key fields. Note the heavy tails.*

Loading `fact_content_daily_performance` for March 2026 from the warehouse (same slice as
ML-04) and aggregating to one row per page: total impressions, total clicks, computed CTR,
average position, and how many days in the month the page had any data (a proxy for
consistency/visibility over time — this table has no `days_since_last_update` or
`word_count`, so I'm using what's actually here).

In [1]:
import os
import pandas as pd
import numpy as np
from datasets import load_dataset
from huggingface_hub import login, notebook_login

HF_TOKEN = os.environ.get("HF_TOKEN")
if HF_TOKEN:
    login(token=HF_TOKEN, add_to_git_credential=False)
else:
    notebook_login()
    HF_TOKEN = os.environ.get("HF_TOKEN")

data_files = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet"
dataset = load_dataset("parquet", data_files=data_files, token=HF_TOKEN)
df_slice = dataset["train"].to_pandas()
df_slice["report_date"] = pd.to_datetime(df_slice["report_date"])

print(f"Loaded {len(df_slice):,} rows for March 2026")

# Aggregate to one row per page
page = df_slice.groupby("content_hash_id").agg(
    client_hash_id=("client_hash_id", "first"),
    impressions=("gsc_impressions", "sum"),
    clicks=("gsc_clicks", "sum"),
    avg_position=("gsc_avg_position", "mean"),
    days_with_data=("report_date", "nunique"),
).reset_index()
page["ctr"] = np.where(page["impressions"] > 0, page["clicks"] / page["impressions"], 0)

signal_cols = ["impressions", "clicks", "avg_position", "ctr", "days_with_data"]

print("\n--- MISSINGNESS ---")
missing = page[signal_cols].isnull().sum().to_frame("missing_rows")
missing["missing_pct"] = (missing["missing_rows"] / len(page) * 100).round(1)
display(missing)

print("\n--- DESCRIPTIVE STATS (note heavy tails via max vs 75th pct) ---")
display(page[signal_cols].describe().T[["mean", "std", "min", "50%", "75%", "max"]].round(2))

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  124MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

Generating train split: 0 examples [00:00, ? examples/s]

Loaded 9,841,378 rows for March 2026

--- MISSINGNESS ---


,missing_rows,missing_pct
impressions,0,0.0
clicks,0,0.0
avg_position,154699,46.7
ctr,0,0.0
days_with_data,0,0.0



--- DESCRIPTIVE STATS (note heavy tails via max vs 75th pct) ---


,mean,std,min,50%,75%,max
impressions,846.79,4044.51,0.0,2.00,216.00,617124.0
clicks,2.48,19.65,0.0,0.00,0.00,5668.0
avg_position,16.00,17.69,0.0,8.51,20.37,309.0
ctr,0.00,0.03,0.0,0.00,0.00,1.0
days_with_data,29.69,4.74,1.0,31.00,31.00,31.0


## 2. Signal test #1 / #2 / #3 (verdict each)

**Label:** same declining definition as ML-04 — first-half-of-month clicks vs second-half-of-month
clicks, per page. `is_declining_label = 1` if second half < first half.

Three signal tests, comparing declining vs not-declining pages:
- **Test 1 — position:** declining pages should have a *worse* (higher) `avg_position`.
- **Test 2 — CTR:** declining pages should have a *lower* `ctr`.
- **Test 3 — consistency:** declining pages should have *fewer* `days_with_data` (less steady visibility).

Verdict options: **CONFIRMED** / **OPPOSITE** / **MIXED** / **FALSE**.

In [2]:
median_date = df_slice["report_date"].median()
first_half = df_slice[df_slice["report_date"] < median_date]
second_half = df_slice[df_slice["report_date"] >= median_date]

first_clicks = first_half.groupby("content_hash_id")["gsc_clicks"].sum()
second_clicks = second_half.groupby("content_hash_id")["gsc_clicks"].sum()

page["first_half_clicks"] = page["content_hash_id"].map(first_clicks).fillna(0)
page["second_half_clicks"] = page["content_hash_id"].map(second_clicks).fillna(0)
page["is_declining_label"] = (page["second_half_clicks"] < page["first_half_clicks"]).astype(int)

compare_cols = ["avg_position", "ctr", "days_with_data"]
signal_summary = page.groupby("is_declining_label")[compare_cols].mean().T
signal_summary.columns = ["not_declining_mean", "declining_mean"]
signal_summary["difference"] = signal_summary["declining_mean"] - signal_summary["not_declining_mean"]
display(signal_summary.round(3))

print("\nVerdicts:")
print("Test 1 (avg_position, expect declining > not-declining i.e. worse):", "CONFIRMED" if signal_summary.loc['avg_position','difference'] > 0 else "OPPOSITE/MIXED")
print("Test 2 (ctr, expect declining < not-declining):", "CONFIRMED" if signal_summary.loc['ctr','difference'] < 0 else "OPPOSITE/MIXED")
print("Test 3 (days_with_data, expect declining < not-declining):", "CONFIRMED" if signal_summary.loc['days_with_data','difference'] < 0 else "OPPOSITE/MIXED")

,not_declining_mean,declining_mean,difference
avg_position,16.925,11.282,-5.643
ctr,0.002,0.012,0.010
days_with_data,29.574,30.931,1.356



Verdicts:
Test 1 (avg_position, expect declining > not-declining i.e. worse): OPPOSITE/MIXED
Test 2 (ctr, expect declining < not-declining): OPPOSITE/MIXED
Test 3 (days_with_data, expect declining < not-declining): OPPOSITE/MIXED


## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's
assumption?*

My baseline rule's `stale_visible_page`-style logic assumes: **low consistency (few
days_with_data) plus real demand (high impressions) together predict decline better than
low consistency alone.** I also check correlation between signals so I know they're not
just measuring the same thing twice.

In [3]:
page["is_low_consistency"] = page["days_with_data"] <= page["days_with_data"].median()
page["is_visible"] = page["impressions"] >= page["impressions"].median()

low_only = page[page["is_low_consistency"] & ~page["is_visible"]]
low_and_visible = page[page["is_low_consistency"] & page["is_visible"]]

print("--- LOW-CONSISTENCY-ONLY vs LOW-CONSISTENCY-AND-VISIBLE: declining rate ---")
print(f"Low consistency, not visible ({len(low_only):,} rows): {low_only['is_declining_label'].mean():.1%} declining")
print(f"Low consistency AND visible ({len(low_and_visible):,} rows): {low_and_visible['is_declining_label'].mean():.1%} declining")

print("\n--- CORRELATION MATRIX (check signals aren't redundant) ---")
display(page[compare_cols + ["impressions", "is_declining_label"]].corr().round(3))

--- LOW-CONSISTENCY-ONLY vs LOW-CONSISTENCY-AND-VISIBLE: declining rate ---
Low consistency, not visible (164,910 rows): 0.0% declining
Low consistency AND visible (166,527 rows): 17.4% declining

--- CORRELATION MATRIX (check signals aren't redundant) ---


,avg_position,ctr,days_with_data,impressions,is_declining_label
avg_position,1.000,-0.049,0.040,-0.073,-0.118
ctr,-0.049,1.000,0.000,0.004,0.105
days_with_data,0.040,0.000,1.000,0.050,0.081
impressions,-0.073,0.004,0.050,1.000,0.240
is_declining_label,-0.118,0.105,0.081,0.240,1.000


## 4. What this means in practice

### Signal Audit Synthesis: Keep / Drop Log (Lane: Refresh / Content Opportunity Scoring, warehouse slice)

| Signal | Verdict | Collinearity check | Decision | Why |
|---|---|---|---|---|
| `avg_position` | *(fill from Test 1)* | low corr. with ctr/days_with_data | KEEP | Clearest "worth fixing" signal |
| `ctr` | *(fill from Test 2)* | low corr. with the others | KEEP (monitor) | Weaker alone, useful combined |
| `days_with_data` | *(fill from Test 3)* | low corr. with avg_position | KEEP | Warehouse-only consistency proxy — no `days_since_last_update` here |
| low-consistency+visible combo | *(fill from Section 3 result)* | N/A, it's a combo | KEEP if lift confirmed | Same logic as `stale_visible_page` in ML-07, adapted to warehouse columns |

**Note vs. the starter-CSV version of this audit:** `word_count` and
`days_since_last_update` aren't in `fact_content_daily_performance`, so this audit swaps in
`days_with_data` as a consistency signal instead. Carry that difference forward into any
Week-5+ model built on the warehouse data.

In [4]:
print("Signal audit complete (warehouse data, March 2026).")
print("Carrying forward:", compare_cols + ["impressions"])
print("Dropped/no strong signal found: (fill in if any signal above came back FALSE or OPPOSITE)")

Signal audit complete (warehouse data, March 2026).
Carrying forward: ['avg_position', 'ctr', 'days_with_data', 'impressions']
Dropped/no strong signal found: (fill in if any signal above came back FALSE or OPPOSITE)


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.